# Elemental Gallium Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarsZDF/gallium/blob/main/demo.ipynb)

**Image gallery grids, experiment tracking, and A/B comparison for image generation workflows.**

This notebook demonstrates the core features of `elemental-gallium`:
- Experiment tracking with SQLite
- Querying and filtering experiments
- Building comparison grids
- A/B comparison
- Export to CSV, JSON, and HTML

## Key Concepts

| Concept | Description |
|---------|-------------|
| **Experiment** | A single image generation run with its metadata (prompt, seed, model, etc.) |
| **Tracker** | SQLite-backed storage for experiments (auto-initialized with `gallium.init()`) |
| **Grid** | Visual comparison of multiple images in a single view |
| **Matrix Grid** | 2D comparison grid organized by two parameters (e.g., model vs seed) |

## Installation

In [ ]:
# Install gallium with grid support
# We pin pillow<12.0 to ensure compatibility with the Colab environment
!pip install -q "pillow<12.0" --no-cache-dir
!pip install -q "elemental-gallium[grid] @ git+https://github.com/MarsZDF/gallium.git@v0.9.4" --no-cache-dir

import gallium
import importlib.metadata

installed_version = importlib.metadata.version("elemental-gallium")
if gallium.__version__ != installed_version:
    print(f"⚠️  Kernel has gallium {gallium.__version__} loaded, but {installed_version} is installed.")
    print(f"⚠️  Please RESTART THE RUNTIME to use the new version! (Runtime > Restart session)")
    raise RuntimeError("Kernel restart required to load updated library.")
else:
    print(f"✅ Gallium {gallium.__version__} ready.")

## Setup

For this demo, we'll create some sample images to work with.

In [ ]:
from PIL import Image
from IPython.display import display, HTML
import os

# Create output directory
os.makedirs("demo_outputs", exist_ok=True)

# Initialize gallium with a fresh database
gallium.init("demo.db")

print(f"Gallium version: {gallium.__version__}")

# Helper function for better visualization
def display_with_info(img, title: str, info: dict = None):
    """Display an image with metadata in a styled container."""
    html = f'<div style="background: #1a1a2e; padding: 15px; border-radius: 8px; margin: 10px 0;">'
    html += f'<h3 style="color: #4ECDC4; margin: 0 0 10px 0;">{title}</h3>'
    if info:
        html += '<div style="color: #ccc; font-family: monospace; font-size: 13px;">'
        for k, v in info.items():
            html += f'<span style="color: #888;">{k}:</span> <span style="color: #fff;">{v}</span>&nbsp;&nbsp;'
        html += '</div>'
    html += '</div>'
    display(HTML(html))
    display(img)

def display_grid_with_title(grid_img, title: str, description: str = None):
    """Display a grid with title and optional description."""
    html = f'<div style="background: #16213e; padding: 15px; border-radius: 8px; margin: 15px 0;">'
    html += f'<h3 style="color: #4ECDC4; margin: 0;">{title}</h3>'
    if description:
        html += f'<p style="color: #aaa; margin: 5px 0 0 0; font-size: 13px;">{description}</p>'
    html += '</div>'
    display(HTML(html))
    display(grid_img)

In [ ]:
# Create sample images (simulating generated images)
def create_sample_image(color: str, size: tuple = (512, 512)) -> Image.Image:
    """Create a simple colored image for demo purposes."""
    return Image.new("RGB", size, color)

# Generate sample "experiments"
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD"]
prompts = [
    "cyberpunk city at night",
    "cyberpunk city at night",
    "forest in morning mist",
    "forest in morning mist",
    "portrait of a robot",
    "portrait of a robot",
]
seeds = [42, 123, 42, 123, 42, 123]
models = ["flux.2-pro", "flux.2-pro", "flux.2-max", "flux.2-max", "flux.2-flex", "flux.2-flex"]
guidance_values = [7.5, 7.5, 5.0, 5.0, 10.0, 10.0]

for i, (color, prompt, seed, model, guidance) in enumerate(zip(colors, prompts, seeds, models, guidance_values)):
    img = create_sample_image(color)
    path = f"demo_outputs/sample_{i}.png"
    img.save(path)
    
    gallium.log(
        prompt=prompt,
        seed=seed,
        path=path,
        model=model,
        width=512,
        height=512,
        duration_ms=1000 + i * 100,
        params={"guidance": guidance, "steps": 28}
    )

print(f"Created {len(colors)} sample experiments")

## 1. Querying Experiments

Use `find()` to query experiments with flexible filters.

### Available Filters

| Filter | Description | Example |
|--------|-------------|---------|
| `id` | Exact experiment ID | `find(id=1)` |
| `prompt` | Exact prompt match | `find(prompt="cyberpunk city")` |
| `prompt__contains` | Prompt contains text | `find(prompt__contains="city")` |
| `prompt__startswith` | Prompt starts with | `find(prompt__startswith="cyber")` |
| `seed` | Exact seed value | `find(seed=42)` |
| `model` | Exact model name | `find(model="flux.2-pro")` |
| `starred` | Starred experiments | `find(starred=True)` |
| `created_after` | Created after date | `find(created_after=datetime(...))` |
| `created_before` | Created before date | `find(created_before=datetime(...))` |

In [ ]:
# Find all experiments
all_experiments = gallium.find()
print(f"Total experiments: {len(all_experiments)}")

# Show experiment details in a formatted table
display(HTML('''
<table style="background: #1a1a2e; color: #fff; border-collapse: collapse; width: 100%; font-size: 13px;">
<tr style="background: #16213e;">
    <th style="padding: 10px; text-align: left; color: #4ECDC4;">ID</th>
    <th style="padding: 10px; text-align: left; color: #4ECDC4;">Prompt</th>
    <th style="padding: 10px; text-align: left; color: #4ECDC4;">Seed</th>
    <th style="padding: 10px; text-align: left; color: #4ECDC4;">Model</th>
    <th style="padding: 10px; text-align: left; color: #4ECDC4;">Guidance</th>
    <th style="padding: 10px; text-align: left; color: #4ECDC4;">Duration</th>
</tr>
''' + ''.join([f'''
<tr style="border-bottom: 1px solid #333;">
    <td style="padding: 8px;">{exp.id}</td>
    <td style="padding: 8px;">{exp.prompt[:30]}...</td>
    <td style="padding: 8px;">{exp.seed}</td>
    <td style="padding: 8px;">{exp.model}</td>
    <td style="padding: 8px;">{exp.params.get("guidance", "N/A")}</td>
    <td style="padding: 8px;">{exp.duration_ms}ms</td>
</tr>''' for exp in all_experiments]) + '</table>'))

In [ ]:
# Filter examples
print("Filter Examples:")
print("-" * 40)

# Filter by prompt content
cyberpunk = gallium.find(prompt__contains="cyberpunk")
print(f"prompt__contains='cyberpunk': {len(cyberpunk)} experiments")

# Filter by seed
seed_42 = gallium.find(seed=42)
print(f"seed=42: {len(seed_42)} experiments")

# Filter by model
flux_pro = gallium.find(model="flux.2-pro")
print(f"model='flux.2-pro': {len(flux_pro)} experiments")

# Filter by ID (new!)
by_id = gallium.find(id=1)
print(f"id=1: {len(by_id)} experiment(s)")

In [ ]:
# Get recent experiments
recent = gallium.recent(3)
print("3 most recent experiments:")
for exp in recent:
    print(f"  [{exp.id}] {exp.prompt[:30]}... (seed={exp.seed})")

## 2. Building Comparison Grids

Create visual grids to compare experiments side-by-side.

In [ ]:
# Basic grid from experiments
all_exp = gallium.find()
grid_img = gallium.grid(all_exp, cols=3, max_size=300)
grid_img.save("demo_outputs/basic_grid.png")

display_grid_with_title(
    grid_img,
    "All Experiments Grid",
    "6 experiments arranged in 3 columns"
)

In [ ]:
# Grid with labels - comparing same prompt across seeds
cyberpunk = gallium.find(prompt__contains="cyberpunk")

# Show the prompt being compared
prompt = cyberpunk[0].prompt if cyberpunk else "N/A"
display(HTML(f'''
<div style="background: #0f3460; padding: 12px; border-radius: 6px; margin-bottom: 10px;">
    <span style="color: #888;">Prompt:</span>
    <span style="color: #fff; font-style: italic;">"{prompt}"</span>
</div>
'''))

labeled_grid = gallium.grid(
    cyberpunk,
    cols=2,
    max_size=400,
    labels=[f"seed={e.seed}" for e in cyberpunk],
    padding=15,
    label_font_size=18,
    background="#1a1a2e"
)
labeled_grid.save("demo_outputs/labeled_grid.png")

display_grid_with_title(
    labeled_grid,
    "Seed Comparison Grid",
    "Same prompt with different seeds - labels show which seed produced each image"
)

## 3. Matrix Grid

Compare experiments across two dimensions (e.g., model vs. seed).

The **matrix grid** automatically organizes experiments into a 2D table where:
- **Rows** = one parameter (e.g., model)
- **Columns** = another parameter (e.g., seed)

This is extremely useful for systematic parameter sweeps!

In [ ]:
# Show what prompts we're comparing
all_exp = gallium.find()
unique_prompts = list(set(e.prompt for e in all_exp))
print("Prompts in comparison:")
for p in unique_prompts:
    print(f"  - {p}")

# Create a matrix comparing models vs seeds
matrix = gallium.matrix_grid(
    all_exp,
    rows="model",
    cols="seed",
    max_size=200,
    show_labels=True,
    label_font_size=16
)
matrix.save("demo_outputs/matrix_grid.png")

display_grid_with_title(
    matrix,
    "Model vs Seed Matrix",
    "Each row is a different FLUX.2 model variant, each column is a different seed"
)

## 4. A/B Comparison

Compare two specific images side-by-side.

In [ ]:
# Compare two experiments
cyberpunk = gallium.find(prompt__contains="cyberpunk")
if len(cyberpunk) >= 2:
    # Show what we're comparing
    display(HTML(f'''
    <div style="background: #0f3460; padding: 12px; border-radius: 6px; margin-bottom: 10px;">
        <div style="color: #4ECDC4; font-weight: bold; margin-bottom: 8px;">A/B Comparison</div>
        <div style="color: #888;">Prompt: <span style="color: #fff; font-style: italic;">"{cyberpunk[0].prompt}"</span></div>
        <div style="color: #888; margin-top: 5px;">
            Comparing seed <span style="color: #FF6B6B;">{cyberpunk[0].seed}</span>
            vs seed <span style="color: #4ECDC4;">{cyberpunk[1].seed}</span>
        </div>
    </div>
    '''))
    
    result = gallium.compare(
        cyberpunk[0].path,
        cyberpunk[1].path,
        labels=(f"seed={cyberpunk[0].seed}", f"seed={cyberpunk[1].seed}")
    )
    comparison = result.grid()
    comparison.save("demo_outputs/comparison.png")
    display(comparison)

## 5. Star and Annotate

Mark your best experiments and add notes.

In [ ]:
# Star the best experiments
best = gallium.find(prompt__contains="cyberpunk", seed=42)
if best:
    gallium.star(best[0].id)
    gallium.annotate(best[0].id, "Best cyberpunk composition")
    print(f"Starred experiment {best[0].id}")

# Find all starred
starred = gallium.find(starred=True)
print(f"\nStarred experiments: {len(starred)}")
for exp in starred:
    print(f"  [{exp.id}] {exp.prompt[:30]}... - {exp.notes}")

## 6. Export Data

Export your experiments to CSV, JSON, or HTML.

In [ ]:
# Export to CSV
gallium.export("csv", path="demo_outputs/experiments.csv")
print("Exported to CSV")

# Export to JSON
gallium.export("json", path="demo_outputs/experiments.json")
print("Exported to JSON")

# Export to HTML gallery
gallium.export("html", path="demo_outputs/gallery.html", title="Demo Gallery")
print("Exported to HTML gallery")

In [ ]:
# View the JSON export
import json

with open("demo_outputs/experiments.json") as f:
    data = json.load(f)
    
print(f"Exported {len(data)} experiments")
print("\nFirst experiment:")
print(json.dumps(data[0], indent=2))

## 7. Cleanup

In [ ]:
# Optional: Clean up demo files
import shutil

# Uncomment to delete demo files
# shutil.rmtree("demo_outputs", ignore_errors=True)
# os.remove("demo.db") if os.path.exists("demo.db") else None
# print("Cleaned up demo files")

## Next Steps

- Check out the [FLUX.2 Demo](flux2_demo.ipynb) for live image generation with BFL API
- See the [examples/](https://github.com/MarsZDF/gallium/tree/main/examples) directory for API integration examples
- Read the full [README](https://github.com/MarsZDF/gallium) for API reference